In [67]:
import pandas as pd
import numpy as np
import firebase_admin
from firebase_admin import credentials, firestore
import joblib

In [68]:
import firebase_admin
from firebase_admin import credentials, firestore

if not firebase_admin._apps:
    cred = credentials.Certificate("smart-agro-ai-fd974-firebase-adminsdk-fbsvc-aa086e000d.json")
    firebase_admin.initialize_app(cred)

db = firestore.client()

In [69]:
docs = db.collection("farm_inputs") \
    .order_by("timestamp", direction=firestore.Query.DESCENDING) \
    .limit(1).stream()

for doc in docs:
    latest_doc_id = doc.id
    data = doc.to_dict()

print("Latest Doc ID:", latest_doc_id)
print(data)

Latest Doc ID: farm1
{'phosphorous': 0.0, 'potassium': 0.0, 'city': '"Pune"', 'moisture': 0.0, 'nitrogen': 0.0, 'timestamp': DatetimeWithNanoseconds(2026, 4, 12, 9, 59, 7, tzinfo=datetime.timezone.utc), 'temperature': 0.0, 'year': 2025, 'rainfall': 288.235238095238}


In [70]:
city = data["city"]
year = data["year"]
temperature = data["temperature"]
moisture = data["moisture"]

nitrogen = data["nitrogen"]
phosphorus = data["phosphorus"]
potassium = data["potassium"]

KeyError: 'phosphorus'

In [ ]:
model = joblib.load("rainfall_model.pkl")

In [ ]:
city_map = {
    "Pune": 0,
    "Mumbai": 1,
    "Delhi": 2,
    "Bangalore": 3,
    "Hyderabad": 4
}

season_map = {
    "Summer": 0,
    "Monsoon": 1,
    "Winter": 2
}

season = "Monsoon"   # later dynamic

rainfall = model.predict([[city_map[city], year, season_map[season]]])[0]

print("Rainfall:", rainfall)

In [ ]:
crop_df = pd.read_csv("crop_recommendation.csv")

In [ ]:
filtered = crop_df[
    (crop_df["nitrogen"] <= nitrogen + 50) &
    (crop_df["phosphorus"] <= phosphorus + 50) &
    (crop_df["potassium"] <= potassium + 50)
]

recommended_crop = filtered.iloc[0]["crop"]

print("Recommended Crop:", recommended_crop)

In [ ]:
doc = db.collection("farm_inputs").document(latest_doc_id)

doc.update({
    "rainfall": float(rainfall),
    "recommended_crop": recommended_crop
})

print("Firebase Updated")